In [ ]:
from dotenv import load_dotenv
load_dotenv()

import json
from datetime import datetime, timedelta
from anthropic import Anthropic
from anthropic.types import Message, ToolParam

client = Anthropic()
model = "claude-sonnet-4-0"

In [ ]:
# --- Helper functions ---
def add_user_message(messages, message):
    messages.append({"role": "user", "content": message.content if isinstance(message, Message) else message})

def add_assistant_message(messages, message):
    messages.append({"role": "assistant", "content": message.content if isinstance(message, Message) else message})

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {"model": model, "max_tokens": 4096, "messages": messages, "temperature": temperature, "stop_sequences": stop_sequences}
    if tools: params["tools"] = tools
    if system: params["system"] = system
    return client.messages.create(**params)

def text_from_message(message):
    return "\n".join(block.text for block in message.content if block.type == "text")

In [ ]:
# --- Tool Functions ---
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


def add_duration_to_datetime(datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"):
    date = datetime.strptime(datetime_str, input_format)
    if unit == "seconds": new_date = date + timedelta(seconds=duration)
    elif unit == "minutes": new_date = date + timedelta(minutes=duration)
    elif unit == "hours": new_date = date + timedelta(hours=duration)
    elif unit == "days": new_date = date + timedelta(days=duration)
    elif unit == "weeks": new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0: month, year = 12, year - 1
        day = min(date.day, [31, 29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31][month - 1])
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years": new_date = date.replace(year=date.year + duration)
    else: raise ValueError(f"Unsupported time unit: {unit}")
    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"  ----\n  Setting the following reminder for {timestamp}:\n  {content}\n  ----")
    return f"Reminder set: '{content}' at {timestamp}"

In [ ]:
# --- Tool Schemas ---
get_current_datetime_schema = ToolParam(
    name="get_current_datetime",
    description="Returns the current date and time formatted according to the specified format string. Use this tool whenever a user asks about the current date, time, or day of the week.",
    input_schema={"type": "object", "properties": {"date_format": {"type": "string", "description": "A Python strftime format string. Defaults to '%Y-%m-%d %H:%M:%S'."}}, "required": []},
)

add_duration_to_datetime_schema = ToolParam(
    name="add_duration_to_datetime",
    description="Adds a specified duration to a datetime string and returns the resulting datetime. Handles seconds, minutes, hours, days, weeks, months, and years.",
    input_schema={"type": "object", "properties": {
        "datetime_str": {"type": "string", "description": "The input datetime string."},
        "duration": {"type": "number", "description": "Amount of time to add."},
        "unit": {"type": "string", "description": "Time unit. Defaults to 'days'."},
        "input_format": {"type": "string", "description": "Format string for parsing. Defaults to '%Y-%m-%d'."},
    }, "required": ["datetime_str"]},
)

set_reminder_schema = ToolParam(
    name="set_reminder",
    description="Creates a timed reminder that will notify the user at the specified time. Use when a user wants to be reminded about something.",
    input_schema={"type": "object", "properties": {
        "content": {"type": "string", "description": "The reminder message text."},
        "timestamp": {"type": "string", "description": "ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS)."},
    }, "required": ["content", "timestamp"]},
)

tools = [get_current_datetime_schema, add_duration_to_datetime_schema, set_reminder_schema]

In [ ]:
# --- Tool routing + conversation loop ---
def run_tool(tool_name, tool_input):
    funcs = {"get_current_datetime": get_current_datetime, "add_duration_to_datetime": add_duration_to_datetime, "set_reminder": set_reminder}
    if tool_name not in funcs: raise ValueError(f"Unknown tool: {tool_name}")
    return funcs[tool_name](**tool_input)

def run_tools(message):
    results = []
    for block in message.content:
        if block.type == "tool_use":
            try:
                result = run_tool(block.name, block.input)
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result), "is_error": False})
                print(f"    {block.name}({block.input}) -> {result}")
            except Exception as e:
                results.append({"type": "tool_result", "tool_use_id": block.id, "content": f"Error: {e}", "is_error": True})
    return results

def run_conversation(messages):
    turn = 0
    while True:
        turn += 1
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)
        text = text_from_message(response)
        if text: print(f"  [Turn {turn}] Claude: {text}")
        if response.stop_reason != "tool_use": break
        print(f"  [Turn {turn}] Tool calls:")
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    return messages

In [ ]:
# Test: Set reminder using multiple tools
messages = []
add_user_message(messages, "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.")
run_conversation(messages)